# 🗂️ Notebook 2: S3 (Object Storage) — Data Model & APIs

In this notebook we build a tiny S3 in pure Python, iterating from a
**naive bad version** to a **reasonable first cut**. The point isn't
to be production-ready — it's to feel *why* each feature exists by
watching the previous version break.

Progression:

1. **Bad:** a dict. Silently overwrites, no metadata, no integrity.
2. **Better:** versioning, tombstones, ETag, typed metadata.
3. **Best (today):** multipart upload for large files.


## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Then in VS Code pick the `.venv` kernel from the top-right of the notebook. If
it doesn't show up: `Cmd+Shift+P` → **Reload Window** and try again.

Everything in this lab is **pure Python** — no databases, no Docker. You can
run it on a laptop in a few seconds.


## 🧱 Entities

- **Bucket** — a namespace and a policy container.
- **Object** — addressed by `(bucket, key, version_id)`, carries a
  blob of bytes plus some metadata (size, ETag, content-type, …).
- **Shard** — a piece of the blob that lives on one storage node.
  We won't implement shards in this notebook; that's Notebook 3.


## 🌐 HTTP-style API

| Method | Path | What it does |
|---|---|---|
| `PUT`    | `/{bucket}` | Create a bucket |
| `PUT`    | `/{bucket}/{key}` | Upload an object |
| `GET`    | `/{bucket}/{key}?versionId=…` | Download an object (or a specific version) |
| `DELETE` | `/{bucket}/{key}` | Delete — creates a **tombstone** if versioning is on |
| `GET`    | `/{bucket}?list&prefix=…` | List objects with a prefix |
| `POST`   | `/{bucket}/{key}?uploads` | Start a multipart upload |
| `PUT`    | `/{bucket}/{key}?partNumber=i&uploadId=…` | Upload one part |
| `POST`   | `/{bucket}/{key}?uploadId=…` | Complete the multipart upload |

Notice the REST shape: buckets and objects are *resources* with URLs,
the verb comes from HTTP. This keeps the API small and familiar.


## 1️⃣ BAD: a naive dict 🙈

Let's start with the simplest thing: a `dict` keyed by `(bucket,
key)`. No versioning, no metadata, no integrity check.


In [ ]:
naive = {}

def bad_put(bucket, key, data):
    naive[(bucket, key)] = data

def bad_get(bucket, key):
    return naive.get((bucket, key))

bad_put("photos", "cat.jpg", b"v1-bytes")
bad_put("photos", "cat.jpg", b"v2-bytes")   # silently overwrites v1!
print("old bytes recoverable? ->", bad_get("photos", "cat.jpg") == b"v1-bytes")
print("there is no record that v1 ever existed.")


**Three concrete problems** with this version:

1. A careless `PUT` erases the previous contents forever.
2. We can't answer "when was this object created?" or "how big is
   it?" without reading the blob.
3. If a disk flips a bit while we're reading, we happily return the
   corrupted bytes.

Let's fix those.


## 2️⃣ BETTER: typed metadata, versioning, ETag 🛡️

We introduce:

- A **pydantic** `ObjectMeta` model — each version gets one.
- **Versioning**: every `PUT` appends a new version; `DELETE` appends
  a **tombstone** (also called a delete marker) instead of erasing.
- **ETag** = MD5 of the bytes. The client and server can both
  compute it and compare — a corrupt read is detectable.


In [ ]:
from datetime import datetime, timezone
from hashlib import md5
from pydantic import BaseModel, Field
from collections import defaultdict
import uuid


class ObjectMeta(BaseModel):
    bucket: str
    key: str
    version_id: str
    size: int = Field(ge=0)
    etag: str
    content_type: str = "application/octet-stream"
    is_delete_marker: bool = False
    created_at: datetime


class VersionedStore:
    def __init__(self):
        # (bucket, key) -> list of ObjectMeta, newest last
        self.meta: dict = defaultdict(list)
        # version_id -> raw bytes (the "data plane")
        self.blobs: dict = {}

    def put(self, bucket, key, data, content_type="application/octet-stream"):
        vid = uuid.uuid4().hex[:8]
        m = ObjectMeta(
            bucket=bucket, key=key, version_id=vid,
            size=len(data), etag=md5(data).hexdigest(),
            content_type=content_type,
            created_at=datetime.now(timezone.utc),
        )
        self.meta[(bucket, key)].append(m)
        self.blobs[vid] = data
        return m

    def delete(self, bucket, key):
        # Tombstone: the bytes stay, but the "latest" is now a delete marker.
        vid = uuid.uuid4().hex[:8]
        m = ObjectMeta(
            bucket=bucket, key=key, version_id=vid,
            size=0, etag="", is_delete_marker=True,
            created_at=datetime.now(timezone.utc),
        )
        self.meta[(bucket, key)].append(m)
        return m

    def get(self, bucket, key, version_id=None):
        # NOTE: `.get(..., [])`, not `self.meta[...]`. `self.meta` is a
        # defaultdict, so indexing it with a missing key *inserts* an empty
        # list — and then `list()` below trips over `versions[-1]` on a key
        # that was only ever read. A read must never mutate the index.
        versions = self.meta.get((bucket, key), [])
        if version_id is not None:
            for m in versions:
                if m.version_id == version_id:
                    if m.is_delete_marker:
                        return None
                    return m, self.blobs.get(m.version_id)
            return None
        if not versions:
            return None
        m = versions[-1]
        if m.is_delete_marker:
            return None
        return m, self.blobs.get(m.version_id)

    def list(self, bucket, prefix="", limit=1000):
        out = []
        for (b, k), versions in sorted(self.meta.items()):   # LIST is key-ordered
            if b != bucket or not k.startswith(prefix) or not versions:
                continue
            latest = versions[-1]
            if latest.is_delete_marker:
                continue
            out.append(latest)
            if len(out) >= limit:
                break
        return out

In [ ]:
# A small tour of VersionedStore
store = VersionedStore()

m1 = store.put("photos", "cat.jpg", b"v1-bytes")
m2 = store.put("photos", "cat.jpg", b"v2-bytes")
store.delete("photos", "cat.jpg")              # <- oops!
store.put("photos", "dog.jpg", b"woof")

print("Latest cat.jpg (after delete):", store.get("photos", "cat.jpg"))

# 🪄 Undelete: just ask for an older version_id.
recovered = store.get("photos", "cat.jpg", m1.version_id)
print("Recovered v1 of cat.jpg:", recovered[1] if recovered else None)

print("List:", [m.key for m in store.list("photos")])


# Regression check for the defaultdict trap: reading a key that does not exist
# must not create an index entry that later breaks LIST.
assert store.get("photos", "no-such-key.jpg") is None
assert store.list("photos"), "LIST must still work after a miss"
print("✅ a GET miss left the index clean")

A delete is now **reversible** as long as the old versions exist —
this is how S3 bucket versioning + MFA-delete protect you from
ransomware and from your own `rm -rf`.


In [ ]:
# ETag catches corruption: a single flipped byte changes the hash.
data = b"important file"
etag = md5(data).hexdigest()
corrupted = data[:-1] + b"?"
print("match?          ", md5(data).hexdigest()      == etag)
print("corrupted match?", md5(corrupted).hexdigest() == etag)


## 3️⃣ BEST (today): multipart upload 🧩

Imagine uploading a **50 GB** video in one HTTP `PUT`. Two things go
wrong:

1. If your connection drops at 90%, you restart at 0%.
2. You can't parallelize — you're bottlenecked by one TCP connection.

The fix is **multipart upload**: the client chops the file into
chunks (5 MB–5 GB each), uploads them independently (in parallel, in
any order), then sends a "complete" request with the list of parts.
The server stitches them together.

Let's first *feel* the pain of the naive approach, then build the
resumable version.


In [ ]:
# BAD: one huge PUT. Every dropped connection restarts from zero.
import random

def bad_upload(size_mb: int, loss_prob_per_mb: float = 0.01) -> int:
    attempts = 0
    while True:
        attempts += 1
        sent = 0
        for _ in range(size_mb):
            if random.random() < loss_prob_per_mb:
                break
            sent += 1
        if sent == size_mb:
            return attempts

random.seed(42)
attempts = bad_upload(500)
print(f"attempts needed for 500 MB single upload: {attempts}")
print(f"bytes actually pushed over the wire     : ~{attempts * 500 / 1024:.0f} GB for a 0.5 GB file")

# With 5 MB parts, only the part that dies is retried.
PART_MB = 5
loss = 0.01
expected_attempts_per_part = 1 / (1 - loss) ** PART_MB      # geometric
mp_mb = 500 * expected_attempts_per_part
print(f"multipart ({PART_MB} MB parts)                  : ~{mp_mb:.0f} MB pushed "
      f"({attempts * 500 / mp_mb:.0f}x less), and each part can go in parallel")

In [ ]:
# BEST: resumable, parallelisable multipart upload — with the rules enforced.
from hashlib import md5
import uuid

MIN_PART = 5 * 1024 * 1024       # S3: every part except the last must be >= 5 MiB
MAX_PARTS = 10_000

class MultipartUpload:
    def __init__(self, bucket, key):
        self.bucket, self.key = bucket, key
        self.upload_id = uuid.uuid4().hex[:8]
        self.parts: dict[int, bytes] = {}      # part_number -> bytes
        self.created_at = 0.0                  # for the GC demo below
        self.aborted = False

    def upload_part(self, part_number: int, data: bytes) -> str:
        if not 1 <= part_number <= MAX_PARTS:
            raise ValueError(f"part number must be 1..{MAX_PARTS}")
        # Re-uploading a part is idempotent by design: the client retries a
        # failed part with the same number and the new bytes simply replace the
        # old. That is why parts are keyed by number and not appended.
        self.parts[part_number] = data
        return md5(data).hexdigest()           # per-part ETag, for the client to verify

    def abort(self):
        """Client gave up. Release the parts — this is not optional, see below."""
        self.parts.clear()
        self.aborted = True

    def complete(self, expected_parts: list[tuple[int, str]] | None = None):
        if not self.parts:
            raise ValueError("no parts uploaded")
        numbers = sorted(self.parts)
        # Rule 1: part numbers must be contiguous from 1. A gap means the client
        # thinks it uploaded something we never received — fail loudly.
        if numbers != list(range(1, len(numbers) + 1)):
            raise ValueError(f"non-contiguous parts: {numbers}")
        # Rule 2: minimum part size, last part exempt. Without this, a client can
        # burn 10,000 parts on a 10 KB object and wreck the metadata plane.
        for n in numbers[:-1]:
            if len(self.parts[n]) < MIN_PART:
                raise ValueError(f"part {n} is {len(self.parts[n])} B, below the {MIN_PART} B minimum")
        # Rule 3: the client sends back the ETags it was given, so a part that
        # silently corrupted in transit is caught here rather than on GET.
        if expected_parts is not None:
            for n, etag in expected_parts:
                if md5(self.parts[n]).hexdigest() != etag:
                    raise ValueError(f"part {n} ETag mismatch — retransmit it")

        ordered = [self.parts[i] for i in numbers]
        blob = b"".join(ordered)
        # Real-S3 multipart ETag: md5 over the concatenation of each part's
        # binary md5, suffixed with "-<numparts>". Note it is NOT the md5 of the
        # object, so you cannot compare it against a locally-hashed file.
        concat = b"".join(md5(p).digest() for p in ordered)
        return blob, f"{md5(concat).hexdigest()}-{len(ordered)}"


mpu = MultipartUpload("videos", "movie.mp4")
payload = b"A" * MIN_PART + b"B" * MIN_PART + b"C" * 1024

e1 = mpu.upload_part(1, payload[:MIN_PART])
e2 = mpu.upload_part(2, payload[MIN_PART:2 * MIN_PART])
e2_retry = mpu.upload_part(2, payload[MIN_PART:2 * MIN_PART])   # network blip → retry
assert e2 == e2_retry, "re-uploading a part must be idempotent"
e3 = mpu.upload_part(3, payload[2 * MIN_PART:])                 # last part may be tiny

blob, etag = mpu.complete(expected_parts=[(1, e1), (2, e2), (3, e3)])
assert blob == payload
print("reconstructed size:", len(blob))
print("multipart ETag    :", etag)

# --- the three ways clients get this wrong -------------------------------
for label, build in [
    ("gap in part numbers",  lambda m: (m.upload_part(1, b"x" * MIN_PART), m.upload_part(3, b"y"))),
    ("undersized mid part",  lambda m: (m.upload_part(1, b"x"), m.upload_part(2, b"y"))),
    ("corrupted part",       lambda m: (m.upload_part(1, b"x" * MIN_PART), m.upload_part(2, b"y"))),
]:
    m = MultipartUpload("videos", "bad.mp4")
    build(m)
    try:
        if label == "corrupted part":
            m.complete(expected_parts=[(1, md5(b"different").hexdigest())])
        else:
            m.complete()
        print(f"❌ {label}: accepted (should not be)")
    except ValueError as e:
        print(f"✅ rejected — {label}: {e}")

### The part of multipart upload that shows up on the invoice

Parts that were uploaded but never `complete`d **still occupy storage**, and
they are invisible: they do not appear in `LIST`, so nobody notices them. A
mobile client that starts a 5 GB upload, loses signal, and never retries leaves
5 GB behind — every time.

This is one of the most common real S3 bills-gone-wrong. The fix is a lifecycle
rule (`AbortIncompleteMultipartUpload`) plus an explicit `abort()` on the client
path. Worth mentioning in an interview: it shows you have operated the thing,
not just designed it.

In [ ]:
# A day in the life of an upload service, with and without garbage collection.
# We account storage *nominally* (one 5 MiB part each) rather than allocating it,
# so this cell stays a few kilobytes instead of a gigabyte.
import random
random.seed(5)

NOMINAL_PART = 5 * 1024 * 1024
pending: dict[str, MultipartUpload] = {}

for day in range(30):
    for _ in range(100):                      # 100 uploads started per day
        m = MultipartUpload("videos", f"clip-{day}")
        m.created_at = float(day)
        m.upload_part(1, b"x")                # stands in for a 5 MiB part
        pending[m.upload_id] = m
        if random.random() < 0.9:             # 90% finish; 10% are abandoned
            m.complete(); del pending[m.upload_id]

held_gb = lambda d: sum(len(m.parts) for m in d.values()) * NOMINAL_PART / 1e9
orphan_gb = held_gb(pending)
print(f"abandoned uploads after 30 days  : {len(pending):,}")
print(f"storage they are silently holding: {orphan_gb:.2f} GB "
      f"(invisible to LIST, fully billed)")

def gc_incomplete(pending, now, older_than_days=7):
    """The lifecycle rule: AbortIncompleteMultipartUpload after N days."""
    dead = [uid for uid, m in pending.items() if now - m.created_at > older_than_days]
    for uid in dead:
        pending[uid].abort(); del pending[uid]
    return len(dead)

reclaimed = gc_incomplete(pending, now=30.0, older_than_days=7)
print(f"GC aborted {reclaimed:,} uploads older than 7 days → {held_gb(pending):.2f} GB remaining")
assert held_gb(pending) < orphan_gb
print()
print("Extrapolate: at 100 M objects/day and a 10% abandon rate this is petabytes")
print("of storage that no API call will ever show you. Set the lifecycle rule on")
print("day one, on every bucket.")

Why this design is lovely:

- **Resumable** — only the failing part retries.
- **Parallel** — the client can push 8 parts at once over 8 TCP
  connections to saturate their pipe.
- **Out-of-order friendly** — parts are keyed by number, not by
  offset, so any ordering works.
- **Self-describing** — the funny "etag-N" format tells any reader
  how many parts the object was uploaded in.


## 🚦 Consistency: one line everybody gets wrong

*After I `PUT` and get a 200, can I `GET` the new bytes immediately?*

- **Same key, same region**: yes. Strong read-after-write for new objects and
  overwrites (since December 2020 for real S3).
- **Listing** after a write: **eventual**. The new object may not appear in a
  `LIST` for a while.
- **Cross-region replication**: **eventual**, typically seconds, occasionally minutes.

Why the split? Because `GET(key)` and `LIST(prefix)` are answered by different
things. A single key maps to exactly one metadata shard, so that shard can
answer authoritatively with no coordination. A prefix listing spans *many*
shards, so it is served from a secondary index that is maintained
asynchronously — and an async index is, by definition, behind.

Let's build both paths and watch them disagree.

In [ ]:

from collections import defaultdict

class MetadataPlane:
    """A key-addressed store (strongly consistent) plus an async LIST index."""

    def __init__(self):
        self.by_key: dict[tuple, dict] = {}          # authoritative, one shard owns each key
        self.list_index: dict[str, set] = defaultdict(set)   # secondary, updated async
        self.index_queue: list[tuple] = []           # the replication lag, made visible

    def put(self, bucket, key, etag):
        self.by_key[(bucket, key)] = {"etag": etag}  # committed on the owning shard
        self.index_queue.append((bucket, key))       # …index update is enqueued, not applied
        return "200 OK"                              # we ack the client HERE

    def get(self, bucket, key):
        return self.by_key.get((bucket, key))        # one shard, no coordination

    def list(self, bucket, prefix=""):
        return sorted(k for k in self.list_index[bucket] if k.startswith(prefix))

    def apply_index_updates(self, n=None):
        """The async indexer catching up. In production: a log tail, seconds behind."""
        todo = self.index_queue[:n] if n else self.index_queue
        for bucket, key in todo:
            self.list_index[bucket].add(key)
        self.index_queue = self.index_queue[len(todo):]
        return len(todo)

mp = MetadataPlane()
mp.put("photos", "cat.jpg", "abc123")

# Read-after-write on the key: always correct, immediately.
assert mp.get("photos", "cat.jpg") == {"etag": "abc123"}
print("GET right after PUT :", mp.get("photos", "cat.jpg"), " ← strongly consistent")

# The same object, via LIST: not there yet.
print("LIST right after PUT:", mp.list("photos"), "                     ← eventually consistent")
assert mp.list("photos") == []

print(f"\nindexer catches up ({mp.apply_index_updates()} update applied)")
print("LIST now            :", mp.list("photos"))
assert mp.list("photos") == ["cat.jpg"]
print()
print("This is the bug behind 'my upload worked but my pipeline did not see it':")
print("a job that discovers work by LIST will miss objects that a GET would")
print("happily return. The fix is never to poll LIST — have the writer emit an")
print("event (S3 Event Notifications → SQS) and drive the pipeline off that.")

## 🔁 Takeaways

- Bad design: "just a dict" loses data and has no observability.
- Fix: typed metadata (pydantic), versioning + tombstones, ETag for
  integrity.
- For large objects, **multipart upload** is non-negotiable. It also
  maps naturally onto erasure-coded shards (Notebook 3).
- Consistency is *always* a tradeoff statement. Write the statement
  down.
